In [3]:
import os
import asyncio
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential
from azure.ai.projects.aio import AIProjectClient
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from dotenv import load_dotenv
import re
import numpy as np
from pypdf import PdfReader
from openai import AsyncAzureOpenAI

load_dotenv()

endpoint_model = os.getenv("AZURE_MODEL_ENDPOINT") # Endpoint do modelo no Azure AI Foundry
deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME") #Nome do deploy do modelo no Azure AI Foundry
model_key = os.getenv("AZURE_AI_MODEL_API_KEY") #Chave de API do modelo no Azure AI Foundry
embedding_deployment = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")

In [5]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "pypdf"])

0

In [6]:
from pypdf import PdfReader
print("Funcionou!")

Funcionou!


In [7]:
# ========================================
# RAG LOCAL COMPLETO (PDF + EMBEDDINGS)
# ========================================
pdf_path = pdf_path = r"D:\Documents\artigo_pos_doc\DO.denisecavalcante_cap1_cap2.pdf"

# ========================================
# 1️⃣ Extrair texto do PDF
# ========================================

def extract_text_from_pdf(path):
    reader = PdfReader(path)
    full_text = ""

    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"

    return full_text


print("📄 Extraindo texto do PDF...")
texto_base = extract_text_from_pdf(pdf_path)
print("✅ Texto extraído!")


# ========================================
# 2️⃣ Cliente de embeddings Azure
# ========================================

embedding_client = AsyncAzureOpenAI(
    azure_endpoint=endpoint_model,
    api_key=model_key,
    api_version="2024-10-21",
)

EMBEDDING_MODEL = embedding_deployment # ou nome do seu deployment


# ========================================
# 3️⃣ Chunking inteligente
# ========================================

def chunk_text(text, max_chars=2200, overlap=300):
    text = re.sub(r"\r\n", "\n", text).strip()
    paragraphs = re.split(r"\n\s*\n", text)

    chunks = []
    current = ""

    for p in paragraphs:
        if len(current) + len(p) < max_chars:
            current += "\n\n" + p
        else:
            chunks.append(current.strip())
            current = p

    if current:
        chunks.append(current.strip())

    final_chunks = []
    for i, chunk in enumerate(chunks):
        if i == 0:
            final_chunks.append(chunk)
        else:
            prev_tail = chunks[i-1][-overlap:]
            final_chunks.append(prev_tail + "\n\n" + chunk)

    return final_chunks


# ========================================
# 4️⃣ Embeddings em batch
# ========================================

async def embed_in_batches(texts, batch_size=16):
    all_vectors = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        response = await embedding_client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch
        )

        vectors = [item.embedding for item in response.data]
        all_vectors.extend(vectors)

    return np.array(all_vectors, dtype=np.float32)


# ========================================
# 5️⃣ Busca por similaridade
# ========================================

def normalize(v):
    norm = np.linalg.norm(v, axis=1, keepdims=True) + 1e-10
    return v / norm

def search_top_k(query_vector, doc_vectors, chunks, k=3):
    doc_norm = normalize(doc_vectors)
    query_norm = query_vector / (np.linalg.norm(query_vector) + 1e-10)

    scores = doc_norm @ query_norm
    idx = np.argsort(scores)[-k:][::-1]

    return [chunks[i] for i in idx]


# ========================================
# 6️⃣ Construir índice RAG
# ========================================

RAG_CHUNKS = []
RAG_VECTORS = None

async def build_rag_index(text):
    global RAG_CHUNKS, RAG_VECTORS

    print("📦 Criando chunks...")
    RAG_CHUNKS = chunk_text(text)
    print(f"🔢 Total de chunks: {len(RAG_CHUNKS)}")

    print("🧠 Gerando embeddings...")
    RAG_VECTORS = await embed_in_batches(RAG_CHUNKS)

    print("✅ Índice RAG pronto!")


async def build_prompt_with_rag(question, top_k=3):
    if RAG_VECTORS is None:
        return question

    query_vec = (
        await embedding_client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=question
        )
    ).data[0].embedding

    query_vec = np.array(query_vec, dtype=np.float32)

    context_chunks = search_top_k(query_vec, RAG_VECTORS, RAG_CHUNKS, k=top_k)
    context = "\n\n---\n\n".join(context_chunks)

    return f"""
Use o CONTEXTO abaixo como base principal da resposta.
Se algo não estiver no contexto, indique que é conhecimento geral.

CONTEXTO:
{context}

PERGUNTA:
{question}
"""


# 🔹 Construir índice agora
await build_rag_index(texto_base)

📄 Extraindo texto do PDF...
✅ Texto extraído!
📦 Criando chunks...
🔢 Total de chunks: 85
🧠 Gerando embeddings...
✅ Índice RAG pronto!


In [8]:
async def criar_agentes():
    """
    Cria agente especializado em revisão e aprimoramento acadêmico.
    """

    chat_client = AzureOpenAIChatClient(
        endpoint=endpoint_model,
        api_key=model_key,
        deployment_name=deployment,  # gpt-35-turbo
        api_version="2024-10-21",
    )

    revisor_agent = ChatAgent(
        chat_client=chat_client,
        name="revisor-academico",
        instructions=(
            "Você é um assistente especialista em revisão acadêmica. "
            "Sua função é corrigir, aprimorar clareza, coesão e formalidade, mantendo o estilo de escrita. "
            "Quando houver CONTEXTO fornecido, utilize-o para enriquecer o texto, "
            "mas sem inventar informações. "
            "Mantenha rigor científico e padrão ABNT."
        ),
    )

    return revisor_agent

In [9]:
import textwrap

async def rodar_agente(paragrafo_usuario):

    revisor_agent = await criar_agentes()

    contexto_enriquecido = await build_prompt_with_rag(paragrafo_usuario)

    prompt_final = f"""
CONTEXTO DO DOCUMENTO:
{contexto_enriquecido}

TEXTO ORIGINAL DO USUÁRIO:
{paragrafo_usuario}

TAREFA:
1. Corrija erros gramaticais.
2. Melhore coesão e clareza.
3. Enriqueça com base no contexto fornecido.
4. Mantenha rigor acadêmico.
5. Não invente informações fora do contexto.
"""

    resposta = await revisor_agent.run(prompt_final)

    # remove quebras estranhas
    texto = " ".join(str(resposta).split())

    # recria parágrafo organizado
    texto_formatado = textwrap.fill(texto, width=90)

    print("=== TEXTO REVISADO ===\n")
    print(texto_formatado)



In [2]:
import sys
print(sys.executable)

c:\Users\Microsoft Windows\agente_ia_academica\.venv\Scripts\python.exe


In [1]:

# 👇 AQUI você coloca o parágrafo que quiser revisar
await rodar_agente("""
  """)

NameError: name 'rodar_agente' is not defined